In [1]:
# Feature Engineering

In [2]:
# For an online retail dataset, the goal during feature engineering is to translate raw transactions into user-level RFM metrics, product-level attributes, and time-series signals.

# We apply two levels of feature engineering:

# 1. Customer-Level Aggregations (RFM)
# 2. Temporal & Behavioral Features


In [3]:
# 1. Customer-Level Aggregations (RFM)

# Aggregate by Customer ID to build customer profiles for segmentation, churn prediction, or lifetime value models.
# Recency: Days since last purchase relative to the dataset snapshot date.
# Frequency: Count of unique Invoice Number items.
# Monetary Value: Calculate total spend per transaction (Quantity \times Unit Price), then sum or average per customer.
# Returns/Cancellations: Count of negative Quantity records, return frequency, and return value ratio.
# Purchase Diversity: Count of unique Stock Code items or categories purchased.
# Inter-purchase Interval: Average, standard deviation, or median days between consecutive orders.

In [4]:
# 2. Temporal & Behavioral Features

# Extract seasonality and behavioral rhythms from Invoice Date.
# Time-of-Day / Day-of-Week: Distribution of orders placed across hours (morning vs. night) or days (weekday vs. weekend).
# Cyclical Transforms: Encode hour (0 to 23) or month (1 to 12) using sine/cosine transformations to preserve continuity.
# Tenure: Days between a customer’s first purchase and their latest purchase.

In [5]:
# Import packages and libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt


In [6]:
# Loading the csv file and check data types
# I call the result dataframe as dff;

dff = pd.read_csv('/content/cleansed_data.csv')

In [7]:
dff.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [8]:
dff.dtypes

,0
InvoiceNo,int64
StockCode,object
Description,object
Quantity,int64
InvoiceDate,object
UnitPrice,float64
CustomerID,float64
Country,object


In [9]:
# Let's convert InvoiceDate from object to datetime:

dff['InvoiceDate'] = pd.to_datetime(dff['InvoiceDate'])
dff.dtypes

,0
InvoiceNo,int64
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
UnitPrice,float64
CustomerID,float64
Country,object


In [10]:
# ---------------------------------------------------------
# 0. Initial Data Prep
# ---------------------------------------------------------
dff["InvoiceDate"] = pd.to_datetime(dff["InvoiceDate"])
dff["Total Line Value"] = dff["Quantity"] * dff["UnitPrice"]

# Reference date for Recency (1 day after the latest transaction)
snapshot_date = dff["InvoiceDate"].max() + pd.Timedelta(days=1)


# ---------------------------------------------------------
# Section 3: Invoice-Level Aggregations (Aggregated to Customer)
# ---------------------------------------------------------
# First, calculate per-invoice metrics
invoice_summary = (
    dff.groupby(["CustomerID", "InvoiceNo"])
    .agg(
        Invoice_Total_Items=("Quantity", "sum"),
        Invoice_Total_Spend=("Total Line Value", "sum"),
        Invoice_Distinct_Products=("StockCode", "nunique"),
        Invoice_Avg_Item_Price=("UnitPrice", "mean"),
        Invoice_Price_Std=("UnitPrice", "std"),
    )
    .reset_index()
)

# Second, average invoice-level behaviors per customer
customer_invoice_features = (
    invoice_summary.groupby("CustomerID")
    .agg(
        Avg_Basket_Size=("Invoice_Total_Items", "mean"),
        Avg_Basket_Value=("Invoice_Total_Spend", "mean"),
        Avg_Basket_Product_Diversity=("Invoice_Distinct_Products", "mean"),
        Avg_Basket_Item_Price=("Invoice_Avg_Item_Price", "mean"),
        Avg_Basket_Price_Variance=("Invoice_Price_Std", "mean"),
    )
    .reset_index()
    .fillna(0)  # Handle NaN variance for single-item baskets
)


# ---------------------------------------------------------
# Section 4: Product-Level Attributes (Aggregated to Customer)
# ---------------------------------------------------------
# First, calculate product popularity metrics across the entire dataset
product_popularity = (
    dff.groupby("StockCode")["Quantity"].sum().to_dict()
)
dff["Product_Popularity_Score"] = dff["StockCode"].map(product_popularity)

# Second, aggregate customer product preferences & price tier interactions
customer_product_features = (
    dff.groupby("CustomerID")
    .agg(
        Min_Item_Price_Paid=("UnitPrice", "min"),
        Max_Item_Price_Paid=("UnitPrice", "max"),
        Median_Item_Price_Paid=("UnitPrice", "median"),
        Avg_Product_Popularity=("Product_Popularity_Score", "mean"),
    )
    .reset_index()
)


# ---------------------------------------------------------
# Sections 1 & 2: Base RFM & Temporal Features
# ---------------------------------------------------------
dff["Hour"] = dff["InvoiceDate"].dt.hour
dff["DayOfWeek"] = dff["InvoiceDate"].dt.dayofweek
dff["Hour_Sin"] = np.sin(2 * np.pi * dff["Hour"] / 24)
dff["Hour_Cos"] = np.cos(2 * np.pi * dff["Hour"] / 24)

customer_rfm_temporal = (
    dff.groupby("CustomerID")
    .agg(
        Recency=(
            "InvoiceDate",
            lambda x: (snapshot_date - x.max()).days,
        ),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("Total Line Value", "sum"),
        Avg_Order_Hour_Sin=("Hour_Sin", "mean"),
        Avg_Order_Hour_Cos=("Hour_Cos", "mean"),
        Weekend_Ratio=("DayOfWeek", lambda x: (x >= 5).mean()),
    )
    .reset_index()
)


# ---------------------------------------------------------
# Consolidated Big Table (ML Feature Matrix)
# ---------------------------------------------------------
final_customer_features = (
    customer_rfm_temporal.merge(
        customer_invoice_features, on="CustomerID", how="left"
    )
    .merge(customer_product_features, on="CustomerID", how="left")
)

# Export the master feature dataset for Machine Learning
final_customer_features.to_csv(
    "final_customer_ml_features.csv", index=False
)

In [11]:
final_customer_features.head()

,CustomerID,Recency,Frequency,Monetary,Avg_Order_Hour_Sin,Avg_Order_Hour_Cos,Weekend_Ratio,Avg_Basket_Size,Avg_Basket_Value,Avg_Basket_Product_Diversity,Avg_Basket_Item_Price,Avg_Basket_Price_Variance,Min_Item_Price_Paid,Max_Item_Price_Paid,Median_Item_Price_Paid,Avg_Product_Popularity
0,12347.0,33,1,711.79,-0.500000,-0.866025,0.0,319.0,711.79,31.0,2.890000,1.505490,0.65,5.95,3.25,381.870968
1,12348.0,24,1,892.80,-0.965926,0.258819,0.0,1254.0,892.80,13.0,2.917647,9.562247,0.29,40.00,0.55,1115.058824
2,12370.0,24,2,1868.02,0.062163,-0.974251,0.0,484.0,934.01,45.0,5.240617,7.224693,0.65,40.00,2.10,290.296703
3,12377.0,21,1,1001.52,0.707107,-0.707107,0.0,604.0,1001.52,43.0,2.106279,1.424465,0.42,4.95,1.95,372.860465
4,12383.0,19,1,600.72,-0.500000,-0.866025,0.0,754.0,600.72,37.0,1.325135,2.421327,0.12,15.00,0.65,554.918919


In [14]:
final_customer_features.shape

(972, 16)

In [15]:
# As it can be seen, the newly made dataframe has 972 rows and 16 columns.
# This is a totally new table and we will be working with this dataframe for machine learning process.

In [16]:
# Feature Engineering is done, and the above dataframe is our final table for machine learning training.
# Next step is finding a suitable machine learning model.